ModuleNotFoundError: No module named 'langchain_chroma'

In [ ]:
pip install langchain langchain-openai langchain-chroma langchain-community langgraph pypdf tiktoken
export OPENAI_API_KEY="AQ.Ab8RN6LYxobf-OOxdVkEnqdgnS2iwQEWVVFHHrvfvFoy3muZMQ"

In [ ]:
import ast
import operator
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# ---------------------------------------------------------
# 1. PDF Loading, Chunking, and Vector Database
# ---------------------------------------------------------
# Load a sample PDF (replace with your actual file path)
# If you don't have one, create a quick text file and use TextLoader instead
loader = PyPDFLoader("financial_report_2026.pdf")
docs = loader.load()

# Split the text into smaller chunks for accurate retrieval
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)

# Store chunks in ChromaDB (runs in-memory for this script)
vectorstore = Chroma.from_documents(documents=chunks, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# ---------------------------------------------------------
# 2. Define Tools for the Agent
# ---------------------------------------------------------

@tool
def search_pdf(query: str) -> str:
    """Search the uploaded PDF for factual information, numbers, or context."""
    results = retriever.invoke(query)
    # Combine the retrieved chunks into a single string for the model
    return "\n\n".join(doc.page_content for doc in results)


def safe_eval(expr: str):
    """A highly secure mathematical evaluator using the AST module."""
    allowed_operators = {
        ast.Add: operator.add, ast.Sub: operator.sub,
        ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.USub: operator.neg, ast.Pow: operator.pow
    }
    
    def _eval(node):
        if isinstance(node, ast.Constant):  # Python 3.8+ handles numbers as Constants
            return node.value
        elif isinstance(node, ast.BinOp):
            left = _eval(node.left)
            right = _eval(node.right)
            return allowed_operators[type(node.op)](left, right)
        elif isinstance(node, ast.UnaryOp):
            return allowed_operators[type(node.op)](_eval(node.operand))
        else:
            raise TypeError(f"Unsupported operation or malicious code detected: {type(node)}")
            
    # Parse the expression strictly as an evaluated math node
    tree = ast.parse(expr, mode='eval').body
    return _eval(tree)


@tool
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression. Input MUST be a valid math string like '25000 * 0.15'."""
    try:
        result = safe_eval(expression)
        return f"Calculation Result: {result}"
    except Exception as e:
        return f"Error evaluating expression: {e}"

# ---------------------------------------------------------
# 3. Build and Invoke the Agent
# ---------------------------------------------------------

def main():
    # Initialize the LLM (the "Brain")
    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    
    # Bundle the tools (the "Hands")
    tools = [search_pdf, calculator]
    
    # create_react_agent automatically builds the Thought -> Action -> Observation loop
    agent = create_react_agent(model, tools=tools)

    # Example Prompt requiring BOTH tools to cooperate
    # The agent will: 
    # 1. Call search_pdf to find the revenue.
    # 2. Call calculator to multiply it by 1.15.
    user_query = "Search the PDF for the Q3 total revenue. Once you find it, calculate what a 15% increase on that revenue would be."
    
    print(f"User: {user_query}\n")
    print("Agent is thinking and using tools...\n")
    
    # Invoke the agent graph
    result = agent.invoke({"messages": [("user", user_query)]})
    
    # Print the final output message from the agent
    print("Final Answer:")
    print(result["messages"][-1].content)

if __name__ == "__main__":
    main()